# Round 6: segment specialist, regularized capacity, stochastic averaging - S6E10

Rounds 3-5 mapped a plateau around 0.9585 for single LightGBM configs on this data (best descriptive
mean 0.95860). This round tests three evidence-driven ideas, still on existing data only:

- **Exp 19: Personal-Travel segment specialist.** The round-5 error analysis found within-segment
  ranking much weaker on Personal Travel (AUC 0.826, n=202k) than elsewhere. CAREFUL: a lower
  within-segment AUC does NOT prove recoverable headroom (the case-mix differs - e.g. only 9.7%
  positives). So this is a MEASURED TEST: does a LightGBM trained only on Personal-Travel rows
  actually rank those rows better than the general model does?
- **Exp 20: regularized high capacity** - Exp 16 (num_leaves=127) was the only positive lead so far
  (4/5 folds positive, within the decision band), so one knob on top of IT: min_data_in_leaf=100,
  to test whether regularizing the larger trees makes the gain reliable.
- **Exp 21/22: stochastic training, then seed averaging.** Exp 21 adds light randomness to the
  incumbent (bagging_fraction=0.9 freq=1, feature_fraction=0.9); Exp 22 averages 3 seeds of that
  config - the standard variance-reduction trick, expecting a small but real gain if variance is
  what separates these plateau configs.

RF note (Andrea's preferred family): the knobs tested so far (leaf size, depth, max_features, ordinal
representation) all landed below LightGBM, though min_samples_leaf=5 did improve RF over its own
default. So this round's three evidence-backed leads go first; RF stays a live option - capacity and
ExtraTrees-style variants return if a measured case earns them, not as blind knob runs.

Reading rules unchanged: paired per-fold deltas, WORKING SIGNAL / NO SIGNAL under the 2x band -
a decision heuristic, not an inferential test. The incumbent is recomputed in-run as always.


In [ ]:
import inspect
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split

RANDOM_STATE = 42
N_SPLITS = 5
CATEGORICAL = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
INCUMBENT_FOLDS_R3 = [0.95851, 0.95751, 0.95907, 0.95841, 0.95885]  # determinism reference
E16_FOLDS = [0.95851, 0.95770, 0.95930, 0.95835, 0.95914]           # Exp 16 (leaves=127) folds


In [ ]:
competition_dir = Path('.')
if not (competition_dir / 'data' / 'train.csv').is_file():
    if Path('playground-series-s6e10/data/train.csv').is_file():
        competition_dir = Path('playground-series-s6e10')
    elif Path('../data/train.csv').is_file():
        competition_dir = Path('..')
data_dir = competition_dir / 'data'

train = pd.read_csv(data_dir / 'train.csv')

def to_binary(series):
    if series.dtype == bool:
        return series.astype(int)
    return series.map({'True': 1, 'False': 0, True: 1, False: 0}).astype(int)

target = to_binary(train['satisfaction'])
features = train.drop(columns=['id', 'satisfaction'])

def to_lgbm(df):
    # LightGBM-native copy: categoricals as 'category', NaN left alone.
    out = df.copy()
    for col in CATEGORICAL:
        out[col] = out[col].astype('category')
    return out

def oof_lgbm(feat_frame, label, name, seed=RANDOM_STATE, return_oof=True, **params):
    frame = to_lgbm(feat_frame)
    oof = np.zeros(len(frame))
    aucs, iters = [], []
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)  # same folds every time
    for fold, (tr_idx, va_idx) in enumerate(skf.split(frame, label), 1):
        X_tr, y_tr = frame.iloc[tr_idx], label.iloc[tr_idx]
        X_va, y_va = frame.iloc[va_idx], label.iloc[va_idx]
        X_fit, X_es, y_fit, y_es = train_test_split(
            X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=RANDOM_STATE)
        # executor fix (disclosed): merge caller params over defaults - the original
        # passed num_leaves twice and crashed Exp 20/21/22 before any new experiment ran.
        cfg = dict(learning_rate=0.05, num_leaves=63, n_estimators=1000,
                   n_jobs=-1, random_state=seed, verbose=-1)
        cfg.update(params)
        model = lgb.LGBMClassifier(**cfg)
        # LightGBM 4.7.0 deprecated eval_set in favour of keyword-only eval_X/eval_y;
        # use whichever signature the installed version supports (no warning either way).
        if 'eval_X' in inspect.signature(lgb.LGBMClassifier.fit).parameters:
            model.fit(X_fit, y_fit, eval_X=X_es, eval_y=y_es, eval_metric='auc',
                      callbacks=[lgb.early_stopping(50, verbose=False)])
        else:
            model.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], eval_metric='auc',
                      callbacks=[lgb.early_stopping(50, verbose=False)])
        oof[va_idx] = model.predict_proba(X_va)[:, 1]
        a = roc_auc_score(y_va, oof[va_idx])
        aucs.append(a); iters.append(model.best_iteration_)
        print(f'  {name} fold {fold}: AUC {a:.5f} (best_iteration={model.best_iteration_})')
    return oof, aucs, iters

print(f'Full train: {len(features):,} rows, {features.shape[1]} raw features')


## Reference: incumbent recompute + within-segment baselines

Same-run incumbent (Exp 8 config), determinism check, then per-fold within-segment AUCs - the paired
baseline for Exp 19.


In [ ]:
print('Reference: incumbent raw LightGBM (Exp 8 config), recomputed in-run')
oof_inc, aucs_inc, iters_inc = oof_lgbm(features, target, 'INC')
diffs = [abs(a - b) for a, b in zip(aucs_inc, INCUMBENT_FOLDS_R3)]
det_note = 'PASS - identical folds' if max(diffs) < 1e-4 else 'WARN - use in-run folds only'
print(f'  determinism vs round-3 constants: max abs diff {max(diffs):.5f} ({det_note})')
INC_FOLDS = aucs_inc
INC_MEAN = float(np.mean(aucs_inc))
print(f'  incumbent mean OOF AUC: {INC_MEAN:.5f} +/- {np.std(aucs_inc):.5f} | pooled: {roc_auc_score(target, oof_inc):.6f}')

# within-segment incumbent AUCs per fold (paired baseline for the specialist)
seg_mask = features['Type of Travel'] == 'Personal Travel'
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
inc_seg_folds = []
for fold, (_, va_idx) in enumerate(skf.split(features, target), 1):
    va_seg = va_idx[seg_mask.iloc[va_idx].to_numpy()]
    inc_seg_folds.append(roc_auc_score(target.iloc[va_seg], oof_inc[va_seg]))
print(f'  incumbent Personal-Travel segment AUCs per fold: ' + ' '.join(f'{a:.5f}' for a in inc_seg_folds))
print(f'  incumbent PT segment mean AUC: {np.mean(inc_seg_folds):.5f} | pooled segment AUC: '
      f'{roc_auc_score(target[seg_mask], oof_inc[seg_mask.to_numpy()]):.6f}')


## Exp 19: Personal-Travel segment specialist

One LightGBM per fold trained ONLY on Personal-Travel rows (same Exp 8 config), scored on the same
fold's Personal-Travel validation rows, paired against the incumbent's within-segment numbers above.
This tests the segment hypothesis directly - it does not assume the weaker within-segment AUC means
recoverable headroom.


In [ ]:
print('Exp 19: LightGBM specialist trained on Personal-Travel rows only')
frame = to_lgbm(features)
oof19 = np.full(len(frame), np.nan)
aucs19 = []
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
for fold, (tr_idx, va_idx) in enumerate(skf.split(frame, target), 1):
    tr_seg = tr_idx[seg_mask.iloc[tr_idx].to_numpy()]
    va_seg = va_idx[seg_mask.iloc[va_idx].to_numpy()]
    X_tr, y_tr = frame.iloc[tr_seg], target.iloc[tr_seg]
    X_fit, X_es, y_fit, y_es = train_test_split(X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=RANDOM_STATE)
    model = lgb.LGBMClassifier(learning_rate=0.05, num_leaves=63, n_estimators=1000,
                               n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
    # LightGBM 4.7.0 deprecated eval_set in favour of keyword-only eval_X/eval_y;
    # use whichever signature the installed version supports (no warning either way).
    if 'eval_X' in inspect.signature(lgb.LGBMClassifier.fit).parameters:
        model.fit(X_fit, y_fit, eval_X=X_es, eval_y=y_es, eval_metric='auc',
                  callbacks=[lgb.early_stopping(50, verbose=False)])
    else:
        model.fit(X_fit, y_fit, eval_set=[(X_es, y_es)], eval_metric='auc',
                  callbacks=[lgb.early_stopping(50, verbose=False)])
    p = model.predict_proba(frame.iloc[va_seg])[:, 1]
    a = roc_auc_score(target.iloc[va_seg], p)
    aucs19.append(a)
    oof19[va_seg] = p
    print(f'  E19 fold {fold}: PT-segment AUC {a:.5f} (n={len(va_seg):,}, best_iteration={model.best_iteration_})')
print(f'Exp 19 PT-segment mean AUC: {np.mean(aucs19):.5f} +/- {np.std(aucs19):.5f}')
seg_deltas = [n - r for n, r in zip(aucs19, inc_seg_folds)]
print(f'  paired segment deltas vs incumbent: ' + ' '.join(f'{d:+.5f}' for d in seg_deltas)
      + f' (mean {np.mean(seg_deltas):+.5f})')


## Exp 20: regularized high capacity (num_leaves=127 + min_data_in_leaf=100)

One knob on top of Exp 16 - the only positive lead so far. Hypothesis: the 127-leaf gain was real but
unreliable because the larger trees overfit; a leaf-size floor should make it consistent. Paired
against BOTH the incumbent and Exp 16's folds.


In [ ]:
print('Exp 20: LightGBM num_leaves=127, min_data_in_leaf=100, lr=0.05')
oof20, aucs20, iters20 = oof_lgbm(features, target, 'E20', num_leaves=127, min_data_in_leaf=100)
print(f'Exp 20 mean OOF AUC: {np.mean(aucs20):.5f} +/- {np.std(aucs20):.5f} | pooled: {roc_auc_score(target, oof20):.6f}')
print('  paired vs incumbent: ' + ' '.join(f'{n-r:+.5f}' for n, r in zip(aucs20, INC_FOLDS)))
print('  paired vs Exp 16:    ' + ' '.join(f'{n-r:+.5f}' for n, r in zip(aucs20, E16_FOLDS)))


## Exp 21: stochastic incumbent (bagging + feature fraction)

One logical change from the incumbent: light randomness (bagging_fraction=0.9, bagging_freq=1,
feature_fraction=0.9). Hypothesis: plateau configs may differ mainly in variance; stochastic training
alone usually costs a little, and Exp 22 tests whether averaging recovers more than it costs.


In [ ]:
print('Exp 21: LightGBM stochastic (bagging 0.9/freq 1, feature_fraction 0.9)')
STOCH = dict(bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9)
oof21, aucs21, iters21 = oof_lgbm(features, target, 'E21', **STOCH)
print(f'Exp 21 mean OOF AUC: {np.mean(aucs21):.5f} +/- {np.std(aucs21):.5f} | pooled: {roc_auc_score(target, oof21):.6f}')


## Exp 22: 3-seed averaging of the stochastic config

Seeds 42/43/44 of the Exp 21 config, OOF probabilities averaged per row. Hypothesis: averaging
independent stochastic runs reduces the variance component without changing the bias - the classic
small-but-real gain. Paired against both Exp 21 (the averaging effect) and the incumbent.


In [ ]:
print('Exp 22: 3-seed average of Exp 21 config')
oof_seeds = []
for seed in [42, 43, 44]:
    o, a, _ = oof_lgbm(features, target, f'E22-s{seed}', seed=seed, **STOCH)
    oof_seeds.append(o)
    print(f'  seed {seed} mean AUC: {np.mean(a):.5f}')
oof22 = np.mean(oof_seeds, axis=0)
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
aucs22 = []
for fold, (_, va_idx) in enumerate(skf.split(features, target), 1):
    a = roc_auc_score(target.iloc[va_idx], oof22[va_idx])
    aucs22.append(a)
    print(f'  E22 fold {fold} (averaged): AUC {a:.5f}')
print(f'Exp 22 mean OOF AUC: {np.mean(aucs22):.5f} +/- {np.std(aucs22):.5f} | pooled: {roc_auc_score(target, oof22):.6f}')


## Results, verdicts, and refit guard

Paired per-fold deltas against the in-run incumbent (Exp 20 also vs Exp 16; Exp 22 also vs Exp 21).
Whole-data winners with a working signal earn a refit; a SEGMENT win (Exp 19) does not earn a
submission file by itself - it earns a combined-pipeline design as its own next experiment.


In [ ]:
def paired_verdict(new_aucs, ref_aucs):
    deltas = [n - r for n, r in zip(new_aucs, ref_aucs)]
    mean_d, spread = float(np.mean(deltas)), float(np.std(deltas))
    if abs(mean_d) <= 2 * spread:
        label = 'NO SIGNAL (within 2x per-fold delta spread - no action)'
    else:
        label = ('WORKING SIGNAL: better' if mean_d > 0 else 'WORKING SIGNAL: worse') + ' (descriptive, not proof)'
    return deltas, mean_d, spread, label

print('Exp 19 PT-segment specialist:')
d, m, s, lab = paired_verdict(aucs19, inc_seg_folds)
print(f'  segment mean {np.mean(aucs19):.5f} vs incumbent segment {np.mean(inc_seg_folds):.5f} | {lab}')

results = {}
for exp_id, name, aucs, oof in [
    ('20', 'LGBM leaves=127+min_data=100', aucs20, oof20),
    ('21', 'LGBM stochastic', aucs21, oof21),
    ('22', 'LGBM stochastic 3-seed avg', aucs22, oof22),
]:
    deltas, mean_d, spread, label = paired_verdict(aucs, INC_FOLDS)
    results[exp_id] = dict(name=name, aucs=aucs, oof=oof, mean_d=mean_d, label=label)
    print(f'Exp {exp_id} {name}: mean {np.mean(aucs):.5f} +/- {np.std(aucs):.5f} | pooled {roc_auc_score(target, oof):.6f}')
    print(f'  paired deltas vs in-run incumbent: ' + ' '.join(f'{x:+.5f}' for x in deltas) + f' (mean {mean_d:+.5f})')
    print(f'  {label}')
d22, m22, s22, lab22 = paired_verdict(aucs22, aucs21)
print(f'Exp 22 vs Exp 21 (the pure averaging effect): mean {m22:+.5f} | {lab22}')
print()

best_oof = max([INC_MEAN] + [float(np.mean(r['aucs'])) for r in results.values()])
print(f'Best OOF mean incl. incumbent: {best_oof:.5f} - >=0.96: {best_oof >= 0.96} (descriptive; target claims stay cautious - repeated selection on this data)')

winners = [k for k, r in results.items()
           if r['mean_d'] > 0 and 'WORKING SIGNAL: better' in r['label']]
if not winners:
    print('No whole-data candidate earned a refit: incumbent Exp 2 raw LightGBM stands, submission_lgbm.csv remains the best submission candidate.')
    if 'WORKING SIGNAL: better' in lab:
        print('Exp 19 segment win noted: earns a combined-pipeline design (general + specialist) as its own experiment.')
else:
    win = max(winners, key=lambda k: np.mean(results[k]['aucs']))
    print(f'Winner: Exp {win} ({results[win]["name"]}) - refits on all rows below; logged as a working signal, exploratory.')


In [ ]:
# Refit ONLY fires on a whole-data working-signal winner over the standing incumbent.
if not winners:
    print('No submission file written this round.')
else:
    test = pd.read_csv(data_dir / 'test.csv')
    test_features = test.drop(columns=['id'])
    if win == '20':
        params, iters = dict(num_leaves=127, min_data_in_leaf=100), iters20
    elif win == '21':
        params, iters = STOCH, iters21
    else:  # '22': average a 3-seed full-data ensemble
        test_probabilities = np.zeros(len(test))
        # best-iteration basis: mean of Exp 21's per-fold best_iterations (seed 42),
        # shared across all three refit seeds - NOT a seed-specific estimate.
        for seed in [42, 43, 44]:
            m = lgb.LGBMClassifier(learning_rate=0.05, num_leaves=63,
                                   n_estimators=int(np.mean(iters21)), n_jobs=-1,
                                   random_state=seed, verbose=-1, **STOCH)
            m.fit(to_lgbm(features), target)
            test_probabilities += m.predict_proba(to_lgbm(test_features))[:, 1] / 3
        params = None
    if params is not None:
        final = lgb.LGBMClassifier(learning_rate=0.05, n_estimators=int(np.mean(iters)),
                                   n_jobs=-1, random_state=RANDOM_STATE, verbose=-1, **params)
        final.fit(to_lgbm(features), target)
        test_probabilities = final.predict_proba(to_lgbm(test_features))[:, 1]
    submission = pd.read_csv(data_dir / 'sample_submission.csv')
    submission['satisfaction'] = test_probabilities
    assert len(submission) == len(test)
    assert submission['id'].equals(test['id'])
    out_path = competition_dir / 'submission_round6.csv'
    submission.to_csv(out_path, index=False)
    print(f"Saved {len(submission):,} predictions to {out_path} - submitting stays Andrea's call.")


In [ ]:
# OOF cache (round-4/5 pattern): persist predictions + metadata for later rounds.
import hashlib, json as _json, time

cache_dir = competition_dir / 'oof_cache'
cache_dir.mkdir(exist_ok=True)
h = hashlib.sha256()
with open(competition_dir / 'data' / 'train.csv', 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 20), b''):
        h.update(chunk)
np.savez(cache_dir / 'round6_oof.npz',
         oof_inc=np.asarray(oof_inc), aucs_inc=np.asarray(aucs_inc), iters_inc=np.asarray(iters_inc),
         oof19=np.asarray(oof19), aucs19=np.asarray(aucs19),
         oof20=np.asarray(oof20), aucs20=np.asarray(aucs20), iters20=np.asarray(iters20),
         oof21=np.asarray(oof21), aucs21=np.asarray(aucs21), iters21=np.asarray(iters21),
         oof22=np.asarray(oof22), aucs22=np.asarray(aucs22),
         seg_mask=np.asarray(seg_mask),
         y=np.asarray(target).astype(int))
meta = {
    'round': 6,
    'protocol': '5-fold stratified OOF, seed 42; LightGBM inner-10% early stop per fold',
    'dataset': {'file': 'data/train.csv', 'rows': int(len(target)), 'sha256': h.hexdigest()},
    'configs': {
        'incumbent': 'LightGBM lr=0.05, num_leaves=63, n_estimators=1000, early stop 50 (Exp 8 config, recomputed in-run)',
        'exp19': 'LightGBM Exp 8 config trained on Personal-Travel rows only; scored within segment',
        'exp20': 'LightGBM lr=0.05, num_leaves=127, min_data_in_leaf=100',
        'exp21': 'LightGBM lr=0.05, num_leaves=63, bagging_fraction=0.9, bagging_freq=1, feature_fraction=0.9',
        'exp22': 'exp21 config, seeds 42/43/44, OOF probabilities averaged',
    },
    'saved_at': time.strftime('%Y-%m-%d %H:%M:%S %Z'),
}
(cache_dir / 'round6_oof.json').write_text(_json.dumps(meta, indent=2))
print('OOF cache written to', cache_dir)


## What happens next

- If Exp 19's specialist wins within segment: build the combined pipeline (general model + PT
  specialist, OOF-stitched) as its own measured experiment - never assumed, always measured.
- If Exp 20 or 22 wins: adopt as reference; remaining on-data levers after that are the
  missing-value handling experiment and heavier averaging (more seeds, or seed x config mixes).
- If everything is null again: the honest report to Andrea is that the approaches tested so far
  haven't reached 0.96 on this data alone - a finite set of configs, not proof it can't be done -
  and the decision that matters is his: try external data (original passenger-survey dataset,
  provenance/terms checked read-only first), more capacity (ExtraTrees-style RF variants, heavier
  ensembling), or accept a strong sub-0.96 model.
